# PySpark reference 9: joins

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-reference-09-joins.ipynb)

The join API, every join type, condition forms, correctness traps, production patterns, physical strategies and skew
handling, each with a runnable example. Practise it in [PySpark practice 9: joins](pyspark-practice-09-joins.ipynb).

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

A local session, the shared `customers`, `orders`, `order_items` and `products` tables (customer 4 has no region,
customers 7 and 8 never ordered, `SKU-4014` has no product row), and three helpers: `join_strategy(df)` runs a query and
lists the join operators of the plan Spark actually used, `no_auto_broadcast()` turns automatic broadcasting off
inside a `with` block so the examples can show the other strategies on tiny data, and `expected_error()` prints the
error class of an example that fails on purpose.

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
import contextlib
import io
import logging
import re
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import Window
from pyspark.sql import functions as F

customers = spark.createDataFrame([
    (1, "Acme Corp", "North", date(2024, 1, 3)),
    (2, "Bright Retail", "North", date(2024, 2, 11)),
    (3, "Cyan Traders", "South", date(2024, 3, 15)),
    (4, "Delta Grocers", None, date(2024, 5, 2)),  # region never arrived from the CRM
    (5, "Echo Supplies", "West", date(2024, 6, 20)),
    (6, "Foxtrot Foods", "West", date(2024, 8, 1)),
    (7, "Gamma Sports", "East", date(2024, 9, 14)),  # 7 and 8 have never ordered
    (8, "Hotel Homeware", "East", date(2024, 11, 5)),
], "customer_id int, name string, region string, signup_date date")

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

order_items = spark.createDataFrame([
    (1, "SKU-4001", 2, D("12.50")), (1, "SKU-4007", 1, D("95.00")),
    (2, "SKU-4002", 1, D("34.50")), (2, "SKU-4009", 2, D("25.00")),
    (3, "SKU-4002", 1, D("30.00")),
    (4, "SKU-4003", 1, D("149.00")), (4, "SKU-4004", 2, D("55.50")),
    (5, "SKU-4005", 1, D("55.25")),
    (6, "SKU-4003", 1, D("149.00")),
    (7, "SKU-4006", 1, D("189.00")), (7, "SKU-4008", 1, D("121.00")),
    (8, "SKU-4010", 1, D("89.90")),
    (9, "SKU-4011", 20, D("59.00")),
    (10, "SKU-4012", 2, D("37.00")),
    (11, "SKU-4013", 1, D("210.00")),
    (12, "SKU-4014", 1, D("41.40")), (12, "SKU-4005", 1, D("55.00")),
], "order_id int, sku string, qty int, unit_price decimal(8,2)")

products = spark.createDataFrame([
    ("SKU-4001", "Kitchen", D("12.50")), ("SKU-4002", "Kitchen", D("34.50")),
    ("SKU-4003", "Garden", D("149.00")), ("SKU-4004", "Garden", D("55.50")),
    ("SKU-4005", "Pantry", D("55.25")), ("SKU-4006", "Outdoor", D("189.00")),
    ("SKU-4007", "Kitchen", D("95.00")), ("SKU-4008", "Outdoor", D("121.00")),
    ("SKU-4009", "Pantry", D("25.00")), ("SKU-4010", "Garden", D("89.90")),
    ("SKU-4011", "Pantry", D("59.00")), ("SKU-4012", "Kitchen", D("37.00")),
    ("SKU-4013", "Outdoor", D("210.00")),  # SKU-4014 is missing
], "sku string, category string, list_price decimal(8,2)")


def plan_lines(df):
    """The lines df.explain() prints."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain()
    return buffer.getvalue().splitlines()


def join_strategy(df):
    """Run df, then return the join operators of the plan Spark actually used (AQE's final plan)."""
    df.collect()
    text = "\n".join(plan_lines(df))
    if "== Final Plan ==" in text:
        text = text.split("== Final Plan ==")[1].split("== Initial Plan ==")[0]
    names = re.findall(r"(BroadcastHashJoin|SortMergeJoin|ShuffledHashJoin|BroadcastNestedLoopJoin|CartesianProduct)", text)
    return names or ["no join operator"]


@contextlib.contextmanager
def expected_error():
    """Run a block that is meant to fail and print the error class instead of a stack trace."""
    spark.sparkContext.setLogLevel("OFF")
    logging.getLogger("DataFrameQueryContextLogger").setLevel(logging.CRITICAL)
    try:
        yield
    except Exception as error:
        print(type(error).__name__ + ":", str(error).split(".")[0][:90])
    finally:
        spark.sparkContext.setLogLevel("ERROR")


@contextlib.contextmanager
def no_auto_broadcast():
    """Disable automatic broadcast joins inside the block, so the examples can show the other strategies."""
    keys = ["spark.sql.autoBroadcastJoinThreshold", "spark.sql.adaptive.autoBroadcastJoinThreshold"]
    old = {k: spark.conf.get(k, None) for k in keys}
    for k in keys:
        spark.conf.set(k, "-1")
    try:
        yield
    finally:
        for k, v in old.items():
            spark.conf.unset(k) if v is None else spark.conf.set(k, v)

## Contents

| section | entries |
| --- | --- |
| [The join API](#the-join-api) | [`DataFrame.join(other, on=None, how=None)`](#dataframejoinother-onnone-hownone), [`DataFrame.crossJoin(other)`](#dataframecrossjoinother) |
| [Join types](#join-types) | [Inner join](#inner-join), [Left outer join](#left-outer-join), [Right outer join](#right-outer-join), [Full outer join](#full-outer-join), [Left semi join](#left-semi-join), [Left anti join](#left-anti-join), [Cross join](#cross-join) |
| [Join conditions](#join-conditions) | [Join on column names](#join-on-column-names), [Join on expressions and aliases](#join-on-expressions-and-aliases), [Null-safe joins](#null-safe-joins), [Non-equi and range joins](#non-equi-and-range-joins), [Self joins](#self-joins) |
| [Column names after a join](#column-names-after-a-join) | [Ambiguous columns and how to resolve them](#ambiguous-columns-and-how-to-resolve-them) |
| [Correctness traps](#correctness-traps) | [Duplicate keys multiply rows](#duplicate-keys-multiply-rows), [Filtering an outer join in WHERE vs ON](#filtering-an-outer-join-in-where-vs-on), [Key type mismatches](#key-type-mismatches), [Nulls introduced by outer joins](#nulls-introduced-by-outer-joins) |
| [Join patterns](#join-patterns) | [Enrich with a lookup and a default](#enrich-with-a-lookup-and-a-default), [Existence checks: semi and anti joins vs `isin`](#existence-checks-semi-and-anti-joins-vs-isin), [Point-in-time (SCD2) joins](#point-in-time-scd2-joins), [As-of joins](#as-of-joins), [Reconcile two datasets](#reconcile-two-datasets) |
| [How Spark executes a join](#how-spark-executes-a-join) | [Choosing a strategy](#choosing-a-strategy), [Broadcast hash join](#broadcast-hash-join), [Sort-merge join](#sort-merge-join), [Shuffle hash join](#shuffle-hash-join), [Broadcast nested loop join and Cartesian product](#broadcast-nested-loop-join-and-cartesian-product), [Join hints](#join-hints) |
| [Skewed joins](#skewed-joins) | [AQE skew join handling](#aqe-skew-join-handling), [Salting a hot key](#salting-a-hot-key) |
| [Joins in Spark SQL](#joins-in-spark-sql) | [SQL join syntax](#sql-join-syntax) |
| [Performance checklist](#performance-checklist) |  |

## The join API

### `DataFrame.join(other, on=None, how=None)`

Joins this DataFrame (the **left** side) with `other` (the **right** side) and returns a new DataFrame. Rows are
combined when the join condition is true; which unmatched rows survive depends on `how`. A join is a
transformation: nothing runs until an action, and Spark picks the physical strategy (broadcast, sort-merge,
shuffle hash, nested loop) when it plans the query.

| parameter | type | description |
| --- | --- | --- |
| `other` | DataFrame | the right side of the join |
| `on` | str, list of str, Column, or list of Columns | **column name(s)** present on both sides (an equi-join that keeps one copy of each key), or a **boolean Column expression** (any condition; both sides' columns are kept). Omitted: a cross join, which Spark only allows through `crossJoin` or a condition |
| `how` | str, default `"inner"` | the join type; see the table below |

| `how` | aliases | keeps |
| --- | --- | --- |
| `"inner"` | | matching pairs only |
| `"left"` | `"leftouter"`, `"left_outer"` | every left row; right columns null when unmatched |
| `"right"` | `"rightouter"`, `"right_outer"` | every right row; left columns null when unmatched |
| `"full"` | `"outer"`, `"fullouter"`, `"full_outer"` | every row from both sides |
| `"left_semi"` | `"semi"`, `"leftsemi"` | left rows that have at least one match, left columns only, never duplicated |
| `"left_anti"` | `"anti"`, `"leftanti"` | left rows with no match, left columns only |
| `"cross"` | | every combination (with no condition) |

**Returns:** a DataFrame. With `on` as names, the key columns appear once, first; with a Column condition, every
column from both sides is kept, including both copies of the key.

**Notes**
- **Null keys never match** in an equality condition, because `null = null` is null, not true. Use
  `eqNullSafe` when nulls should match ([null-safe joins](#null-safe-joins)).
- A key that repeats on both sides produces **one row per pair**: 3 matching rows on the left and 2 on the right
  give 6 output rows ([duplicate keys](#duplicate-keys-multiply-rows)).
- The output row order is not defined. Sort explicitly.

In [ ]:
# Name-based equi-join: one customer_id column in the output.
orders.join(customers, "customer_id").select("order_id", "customer_id", "name", "amount").orderBy("order_id").show(4)

# Expression join: both customer_id columns are kept, so qualify them through aliases.
o, c = orders.alias("o"), customers.alias("c")
o.join(c, F.col("o.customer_id") == F.col("c.customer_id"), "inner").select("o.order_id", "c.customer_id", "c.name").orderBy("order_id").show(4)

### `DataFrame.crossJoin(other)`

Returns the Cartesian product: every left row paired with every right row, `left_rows × right_rows` rows in total.
It takes no condition. Spark refuses an implicit cross join (a `join` with no usable condition) unless you call
`crossJoin` or use `how="cross"`, so a missing condition fails loudly instead of exploding silently.

| parameter | type | description |
| --- | --- | --- |
| `other` | DataFrame | the right side |

**Returns:** a DataFrame with every column of both sides.

**Notes:** use it to build a **scaffold** of all combinations (every customer × every day, every store × every
product) that a later left join fills in. Keep both sides small: 10,000 × 10,000 rows is 100 million.

In [ ]:
days = spark.createDataFrame([(date(2026, 1, 5),), (date(2026, 1, 6),)], "day date")
scaffold = customers.where("customer_id <= 3").select("customer_id").crossJoin(days)
print("rows:", scaffold.count())  # 3 customers x 2 days
daily = orders.groupBy("customer_id", F.to_date("order_ts").alias("day")).agg(F.count("*").alias("orders"))
scaffold.join(daily, ["customer_id", "day"], "left").fillna(0, ["orders"]).orderBy("customer_id", "day").show()

## Join types

The examples in this section join two small tables so every row is visible: `left_t` has keys 1, 2, 3 and
`right_t` has keys 2, 3, 3, 4. Key 3 appears twice on the right, and keys 1 and 4 exist on only one side.

In [ ]:
left_t = spark.createDataFrame([(1, "a"), (2, "b"), (3, "c")], "k int, left_val string")
right_t = spark.createDataFrame([(2, "x"), (3, "y"), (3, "z"), (4, "w")], "k int, right_val string")

### Inner join

Returns one row for every pair of left and right rows whose condition is true. Rows with no match on the other
side are dropped from both sides. This is the default `how`.

**Use it for:** combining facts with dimensions when every fact must have a dimension row, and when unmatched rows
are genuinely not wanted.

**Notes:** an inner join silently **drops** facts whose dimension row is missing, which is the most common cause of
"revenue went down after the join". Count rows before and after, or use a left join and count the nulls.

In [ ]:
left_t.join(right_t, "k", "inner").orderBy("k", "right_val").show()

### Left outer join

Returns every left row. Where a right row matches, its columns are filled in; where none matches, the right
columns are null. Left rows with several matches appear once per match.

**Use it for:** enrichment that must not lose facts: orders with customer attributes, events with device details.
It is the default choice for joining a fact table to a dimension.

**Notes:** a left join can still **increase** the row count when the right key is not unique. Filtering on a
right-side column in `where` afterwards discards the null rows and quietly turns the join into an inner join
([WHERE vs ON](#filtering-an-outer-join-in-where-vs-on)).

In [ ]:
left_t.join(right_t, "k", "left").orderBy("k", "right_val").show()

### Right outer join

The mirror of a left join: every right row is kept, and left columns are null where nothing matches.

**Use it for:** rarely needed; swapping the two sides and using a left join reads more naturally and is what most
teams standardize on.

In [ ]:
left_t.join(right_t, "k", "right").orderBy("k", "right_val").show()

### Full outer join

Returns every row from both sides: matched pairs, left-only rows (right columns null) and right-only rows (left
columns null). With `on` as a name, Spark merges the key into one non-null column; with an expression condition,
you get both key columns and must `coalesce` them.

**Use it for:** reconciling two datasets (yesterday's vs today's snapshot, source vs target counts), and merging
two partial feeds into one.

**Notes:** a full outer join cannot be broadcast on either side, so it always shuffles both inputs.

In [ ]:
left_t.join(right_t, "k", "full").orderBy("k", "right_val").show()

l, r = left_t.alias("l"), right_t.alias("r")
l.join(r, F.col("l.k") == F.col("r.k"), "full").select(
    F.coalesce("l.k", "r.k").alias("k"), F.col("l.k").alias("left_k"), F.col("r.k").alias("right_k"), "left_val", "right_val"
).orderBy("k", "right_val").show()

### Left semi join

Returns the left rows that have **at least one** match on the right, with **only the left columns**. Each left row
appears at most once, however many matches it has. It answers "which rows exist in the other table?", SQL's
`WHERE EXISTS` / `IN (subquery)`.

**Use it for:** filtering a large table by membership in another (orders of active customers, events for flagged
devices) without duplicating rows or carrying unwanted columns.

**Notes:** an inner join followed by `select(left columns).distinct()` gives the same rows but does more work and
also removes genuine duplicate rows from the left side. Prefer the semi join.

In [ ]:
left_t.join(right_t, "k", "left_semi").orderBy("k").show()
print("inner join rows for the same keys:", left_t.join(right_t, "k").count())

### Left anti join

Returns the left rows that have **no** match on the right, with only the left columns. It answers "what is
missing?", SQL's `WHERE NOT EXISTS`.

**Use it for:** customers who never ordered, orphaned facts whose dimension row is missing (referential-integrity
checks), new records not yet loaded into a target table.

**Notes:** unlike SQL's `NOT IN`, an anti join is not broken by nulls on the right side: a null right key simply
matches nothing. A left row whose own key is null is always returned, because it can never match.

In [ ]:
left_t.join(right_t, "k", "left_anti").orderBy("k").show()
customers.join(orders, "customer_id", "left_anti").select("customer_id", "name").show()

### Cross join

Pairs every left row with every right row. See [`crossJoin`](#dataframecrossjoinother) for the API; the example
shows the size of the result.

**Notes:** a join whose condition compares no columns from both sides (for example a filter on one side only) is
also a cross join and will be planned as a Cartesian product or broadcast nested loop join.

In [ ]:
print("cross join rows:", left_t.crossJoin(right_t).count(), "=", left_t.count(), "x", right_t.count())

## Join conditions

### Join on column names

`on="key"` or `on=["k1", "k2"]` builds an equality condition on columns that have the **same name on both sides**,
and keeps a single copy of each key column in the output (first in the column list). For a full outer join the
merged key is the non-null value from either side.

**Use it for:** most fact-to-dimension joins, where keys share a name. It avoids ambiguous-column errors entirely.

**Notes:** the key columns must exist on both sides with the same name; rename one side first
(`withColumnRenamed`) when they differ. A list of names means **all** of them must be equal (AND).

In [ ]:
order_items.join(products, "sku", "left").select("order_id", "sku", "category").orderBy("order_id", "sku").show(4)
daily.join(daily.withColumnRenamed("orders", "orders_again"), ["customer_id", "day"]).orderBy("customer_id", "day").show(3)

### Join on expressions and aliases

Pass a boolean Column as `on` for any condition: different column names, several conditions combined with `&`
and `|`, inequalities or function calls. Both sides' columns are kept, so give each DataFrame an alias with
`df.alias("o")` and refer to columns as `F.col("o.customer_id")`. `df["col"]` also works but breaks in self joins,
where both sides come from the same DataFrame.

**Use it for:** keys with different names, composite keys with extra rules, and any non-equality join.

**Notes:** wrap each comparison in parentheses, because `&` binds more tightly than `==` in Python.

In [ ]:
o, i = orders.alias("o"), order_items.alias("i")
condition = (F.col("o.order_id") == F.col("i.order_id")) & (F.col("i.qty") >= 2)
o.join(i, condition).select("o.order_id", "o.status", "i.sku", "i.qty").orderBy("order_id", "sku").show()

### Null-safe joins

`a == b` is null (not true) when either side is null, so rows with null keys never match. `a.eqNullSafe(b)`
(SQL `a <=> b`) treats two nulls as equal and a null against a value as not equal.

| left key | right key | `==` | `eqNullSafe` |
| --- | --- | --- | --- |
| 1 | 1 | true | true |
| 1 | null | null | false |
| null | null | null | **true** |

**Use it for:** composite keys where null is a meaningful value ("all regions", "no promotion"), and comparing two
versions of a table where both sides may hold null.

**Notes:** Spark cannot use a null-safe condition with every optimization that plain equality allows, but it is
still planned as an equi-join (hash or sort-merge).

In [ ]:
targets = spark.createDataFrame([("DE", "Berlin", 100), ("DE", None, 500)], "country string, region string, target int")
actuals = spark.createDataFrame([("DE", "Berlin", 90), ("DE", None, 480)], "country string, region string, actual int")
t, a = targets.alias("t"), actuals.alias("a")
plain = t.join(a, (F.col("t.country") == F.col("a.country")) & (F.col("t.region") == F.col("a.region")))
safe = t.join(a, (F.col("t.country") == F.col("a.country")) & F.col("t.region").eqNullSafe(F.col("a.region")))
print("plain equality rows:", plain.count(), "| null-safe rows:", safe.count())
safe.select("t.country", "t.region", "target", "actual").orderBy(F.col("region").asc_nulls_first()).show()

### Non-equi and range joins

A condition can use `<`, `<=`, `between` or any boolean expression. Typical cases: a value falls in a band, a
timestamp falls in a validity interval, an event happens within N minutes of another.

**Use it for:** price bands, SCD2 validity intervals ([point-in-time joins](#point-in-time-scd2-joins)), "events
within 30 minutes of a click".

**Notes:** with **no equality** in the condition, Spark cannot hash or sort by a key and falls back to a broadcast
nested loop join (or a Cartesian product), which compares every pair. Add an equality on a coarse key whenever you
can (same customer, same day bucket), so the range check runs only within matching keys.

In [ ]:
bands = spark.createDataFrame([("small", 0, 100), ("medium", 100, 500), ("large", 500, 100000)], "band string, low int, high int")
banded = orders.join(bands, (orders.amount >= bands.low) & (orders.amount < bands.high))
banded.select("order_id", "amount", "band").orderBy("order_id").show(5)
print("strategy:", join_strategy(banded))

### Self joins

A DataFrame joined to itself: employees to their managers, a product to related products, pairs of customers.
Alias each side (`df.alias("e")`, `df.alias("m")`) and refer to columns only through the aliases.

**Use it for:** hierarchies one level at a time, pair generation (`a.id < b.id` to avoid mirrored duplicates and
self-pairs), and comparing rows of the same table.

**Notes:** `df["col"]` cannot tell the two sides apart in a self join and raises an ambiguity error or silently
compares a column with itself; always use aliases.

In [ ]:
staff = spark.createDataFrame([(1, "Ana", None), (2, "Ben", 1), (3, "Cai", 1), (4, "Dee", 2)], "emp_id int, name string, manager_id int")
e, m = staff.alias("e"), staff.alias("m")
e.join(m, F.col("e.manager_id") == F.col("m.emp_id"), "left").select(
    F.col("e.name").alias("employee"), F.col("m.name").alias("manager")).orderBy("employee").show()

c1, c2 = customers.alias("c1"), customers.alias("c2")
c1.join(c2, (F.col("c1.region") == F.col("c2.region")) & (F.col("c1.customer_id") < F.col("c2.customer_id"))).select(
    "c1.region", F.col("c1.name").alias("customer_a"), F.col("c2.name").alias("customer_b")).orderBy("region").show(truncate=False)

## Column names after a join

### Ambiguous columns and how to resolve them

When both sides have a column with the same name and the join uses an expression (not `on=` names), the output
holds **two columns with that name**. Referring to it by name raises `AMBIGUOUS_REFERENCE`. Ways to resolve it:

| approach | when |
| --- | --- |
| join `on="key"` names | the shared columns are the keys |
| `df.alias("x")` + `F.col("x.col")` | you need columns from both sides |
| rename before joining (`withColumnRenamed`, `withColumnsRenamed`, or `select` with `alias`) | non-key columns share names (`status`, `updated_at`) |
| `drop(right_df.col)` after the join | you want only one side's copy |

**Notes:** select an explicit column list after every join in production code. It documents the output schema and
stops new upstream columns from colliding later.

In [ ]:
shipments = spark.createDataFrame([(1, "delivered"), (2, "in_transit")], "order_id int, status string")
joined = orders.join(shipments, orders.order_id == shipments.order_id)
with expected_error():
    joined.select("status").show()

joined.select(orders.order_id, orders.status.alias("order_status"), shipments.status.alias("shipment_status")).show()
orders.join(shipments.withColumnRenamed("status", "shipment_status"), "order_id").select("order_id", "status", "shipment_status").show()

## Correctness traps

### Duplicate keys multiply rows

Every left row is paired with **every** matching right row. If the dimension you join to has two rows for a key
(two versions of a customer, a product listed in two catalogs), every fact for that key is duplicated, and every
sum after the join is inflated. This is the most common silent join bug.

**How to catch it:** check the key is unique on the side that should be unique, and compare row counts before and
after the join.

```python
dupes = dim.groupBy("key").count().where("count > 1")   # must be empty
assert facts.join(dim, "key", "left").count() == facts.count()
```

**How to fix it:** deduplicate the dimension to one row per key first (latest version with `row_number`, current
row of an SCD2 table, or `max_by`), or aggregate the many side before joining.

In [ ]:
regions_v = spark.createDataFrame([(1, "North", 1), (1, "West", 2), (2, "North", 1)], "customer_id int, region string, version int")
print("orders of customers 1-2:", orders.where("customer_id <= 2").count())
print("after joining all versions:", orders.where("customer_id <= 2").join(regions_v, "customer_id").count())
regions_v.groupBy("customer_id").count().where("count > 1").show()

latest = regions_v.withColumn("rn", F.row_number().over(Window.partitionBy("customer_id").orderBy(F.desc("version")))).where("rn = 1").drop("rn")
print("after joining the latest version:", orders.where("customer_id <= 2").join(latest, "customer_id").count())

### Filtering an outer join in WHERE vs ON

For an **outer** join, a condition on the optional side means different things in the join condition and in a
later filter:

- In the **join condition** (`on`), it decides which right rows are allowed to match; left rows without an allowed
  match are **kept** with nulls.
- In a **`where` after the join**, it runs on the joined rows; left rows whose right columns are null fail the
  test and are **removed**, so the left join behaves like an inner join.

For an inner join the two placements give the same result.

**Use it for:** reports that must list every customer while counting only some of their orders ("paid orders per
customer, including customers with none").

In [ ]:
c, o = customers.alias("c"), orders.alias("o")
in_on = c.join(o, (F.col("c.customer_id") == F.col("o.customer_id")) & (F.col("o.status") == "paid"), "left")
in_where = c.join(o, F.col("c.customer_id") == F.col("o.customer_id"), "left").where(F.col("o.status") == "paid")
print("condition in ON:   ", in_on.select("c.customer_id").distinct().count(), "customers")
print("condition in WHERE:", in_where.select("c.customer_id").distinct().count(), "customers")
in_on.groupBy("c.customer_id").agg(F.count("o.order_id").alias("paid_orders")).orderBy("customer_id").show()

### Key type mismatches

If the key is an `int` on one side and a `string` on the other, Spark inserts an implicit cast of the string side to
a number so the comparison can run. Under Spark 4's default **ANSI mode**, one value that does not convert (`"C-3"`)
**fails the whole job** with `CAST_INVALID_INPUT`, at run time, possibly hours into a pipeline. With ANSI mode off,
the same value silently fails to match instead. Either way, the implicit cast also hides the key from optimizations
such as bucketing.

**How to avoid it:** clean and cast explicitly, once, on the side you control (`trim`, strip prefixes,
`try_cast`), and count the keys that failed to convert before joining.

In [ ]:
crm = spark.createDataFrame([("1", "gold"), (" 2", "silver"), ("C-3", "bronze")], "customer_id string, tier string")
with expected_error():
    customers.join(crm, customers.customer_id == crm.customer_id).count()

clean = crm.withColumn("customer_id", F.regexp_replace(F.trim("customer_id"), "^C-", "").try_cast("int"))
print("keys that failed to convert:", clean.where("customer_id IS NULL").count())
print("matches after cleaning:", customers.join(clean, "customer_id").count())

### Nulls introduced by outer joins

Outer joins create nulls in the optional side's columns. They change later logic:

- `count(col)` skips them, `count("*")` does not;
- `sum` and `avg` skip them, and a sum over only nulls is null rather than 0;
- `col != "x"` is null for them, so a filter drops them.

Decide for each column whether a missing match means 0, "unknown" or "exclude", and apply it with `coalesce` or
`fillna` straight after the join.

In [ ]:
per_customer = customers.join(orders, "customer_id", "left").groupBy("customer_id").agg(
    F.count("*").alias("count_star"), F.count("order_id").alias("count_orders"), F.sum("amount").alias("revenue"))
per_customer.withColumn("revenue_or_zero", F.coalesce("revenue", F.lit(0))).where("customer_id >= 6").orderBy("customer_id").show()

## Join patterns

### Enrich with a lookup and a default

Left join the facts to a lookup, then `coalesce` the looked-up value with a default, so unmatched rows carry an
explicit value ("unknown", 0) instead of null and no fact is lost. Broadcast the lookup when it is small.

**Use it for:** categories, regions, currency names, status labels.

In [ ]:
lines = order_items.join(F.broadcast(products.select("sku", "category")), "sku", "left").withColumn(
    "category", F.coalesce("category", F.lit("unknown")))
lines.groupBy("category").agg(F.sum(F.col("qty") * F.col("unit_price")).alias("revenue")).orderBy("category").show()

### Existence checks: semi and anti joins vs `isin`

To keep rows whose key exists (or does not exist) in another DataFrame, use a semi (or anti) join. The
alternative, collecting the keys to the driver and filtering with `isin(list)`, only works for small key sets,
moves data through the driver and bakes a literal list into the plan.

| need | use |
| --- | --- |
| keys from another DataFrame | `left_semi` / `left_anti` join |
| a short fixed list known in code | `F.col(k).isin("a", "b")` |
| the same in SQL | `WHERE EXISTS` / `WHERE NOT EXISTS` subqueries, or `LEFT SEMI JOIN` / `LEFT ANTI JOIN` |

In [ ]:
returners = orders.where("status = 'returned'").select("customer_id")
customers.join(returners, "customer_id", "left_semi").select("customer_id", "name").orderBy("customer_id").show()
spark.sql("SELECT customer_id, name FROM {c} c WHERE NOT EXISTS (SELECT 1 FROM {o} o WHERE o.customer_id = c.customer_id) ORDER BY 1",
          c=customers, o=orders).show()

### Point-in-time (SCD2) joins

A type 2 dimension keeps one row per version of an entity with a validity interval `[valid_from, valid_to)`, where
the current row has `valid_to` null. To attach the version that was valid **when the fact happened**, join on the
entity key **and** the interval:

```python
(f.key == d.key) & (f.ts >= d.valid_from) & (f.ts < coalesce(d.valid_to, far_future))
```

**Notes:** use half-open intervals (`>=` start, `<` end) so a timestamp on a boundary matches exactly one version.
Use a left join so facts outside every interval stay visible (as nulls) instead of disappearing.

In [ ]:
segments = spark.createDataFrame(
    [(1, "standard", datetime(2024, 1, 1), datetime(2026, 1, 5, 10, 0)), (1, "premium", datetime(2026, 1, 5, 10, 0), None),
     (2, "standard", datetime(2026, 1, 7), None)],
    "customer_id int, segment string, valid_from timestamp, valid_to timestamp")
o, s = orders.where("customer_id <= 2").alias("o"), segments.alias("s")
far_future = F.lit("9999-12-31").cast("timestamp")
cond = (F.col("o.customer_id") == F.col("s.customer_id")) & (F.col("o.order_ts") >= F.col("s.valid_from")) & (
    F.col("o.order_ts") < F.coalesce(F.col("s.valid_to"), far_future))
o.join(s, cond, "left").select("o.order_id", "o.order_ts", "s.segment").orderBy("order_id").show()

### As-of joins

An **as-of join** attaches, for each left row, the latest right row at or before the left row's time: the exchange
rate in force at the order time, the last known location before an event. PySpark has no dedicated operator. Two
common implementations:

1. **Range join + window:** join on the key with `right.ts <= left.ts`, then keep the latest match per left row
   with `row_number` ordered by `right.ts` descending.
2. **Validity intervals:** derive `valid_to` with `lead(ts)` on the right side, then do a
   [point-in-time join](#point-in-time-scd2-joins). This avoids the many-matches intermediate result.

**Notes:** approach 1 creates every earlier match before filtering, which can be large; approach 2 scales better.
pandas-on-Spark has `merge_asof` for small data.

In [ ]:
fx = spark.createDataFrame(
    [("EUR", datetime(2026, 1, 4), 1.08), ("EUR", datetime(2026, 1, 7), 1.10), ("EUR", datetime(2026, 1, 12, 12), 1.09)],
    "currency string, rate_ts timestamp, usd_rate double")
o = orders.withColumn("currency", F.lit("EUR")).alias("o")
valid = fx.withColumn("valid_to", F.lead("rate_ts").over(Window.partitionBy("currency").orderBy("rate_ts"))).alias("r")
asof = o.join(valid, (F.col("o.currency") == F.col("r.currency")) & (F.col("o.order_ts") >= F.col("r.rate_ts")) & (
    F.col("o.order_ts") < F.coalesce(F.col("r.valid_to"), far_future)), "left")
asof.select("o.order_id", "o.order_ts", "r.rate_ts", "r.usd_rate").orderBy("order_id").show(12)

### Reconcile two datasets

A full outer join on the business key lines up two versions of a dataset (yesterday's and today's snapshot, source
and target). Classify each key by which side it appears on and whether its values changed. Compare values with
`eqNullSafe` (or a hash of all non-key columns) so null-to-null is not reported as a change.

In [ ]:
yesterday = spark.createDataFrame([(1, "paid"), (2, "placed"), (3, "shipped")], "order_id int, status string")
today = spark.createDataFrame([(2, "paid"), (3, "shipped"), (4, "placed")], "order_id int, status string")
y, t = yesterday.alias("y"), today.alias("t")
diff = y.join(t, F.col("y.order_id") == F.col("t.order_id"), "full").select(
    F.coalesce("y.order_id", "t.order_id").alias("order_id"),
    F.col("y.status").alias("old_status"), F.col("t.status").alias("new_status"),
    F.when(F.col("y.order_id").isNull(), "inserted").when(F.col("t.order_id").isNull(), "deleted")
     .when(~F.col("y.status").eqNullSafe(F.col("t.status")), "updated").otherwise("unchanged").alias("change"))
diff.orderBy("order_id").show()

## How Spark executes a join

### Choosing a strategy

Spark picks a physical join strategy from the join type, the condition and the estimated size of each side. Adaptive
Query Execution (AQE, on by default) can change the choice at run time once it knows the real sizes.

| strategy | requires | cost | chosen when |
| --- | --- | --- | --- |
| **Broadcast hash join** | an equality condition; the broadcast side small | no shuffle of the big side | one side is below `spark.sql.autoBroadcastJoinThreshold` (10 MB by default), or hinted |
| **Shuffle hash join** | an equality condition | shuffles both sides, builds a hash table per partition | hinted, or preferred when one side is much smaller but too big to broadcast |
| **Sort-merge join** | an equality condition, sortable keys | shuffles and sorts both sides | the default for large equi-joins |
| **Broadcast nested loop join** | nothing | compares every pair; one side broadcast | non-equi conditions with a small side |
| **Cartesian product** | nothing | every pair, both sides large | non-equi conditions with no small side |

Which side can be broadcast depends on the join type: the right side of a left join, the left side of a right join,
either side of an inner join, and neither side of a full outer join.

**Planned vs actual strategy.** `explain()` before an action shows the plan built from size **estimates**. A
DataFrame made from local Python data, or read through a source without statistics, has no useful estimate, so the
initial plan is often a sort-merge join. When the query runs, AQE measures the real size of each shuffle and can
replace the sort-merge join with a broadcast hash join. `explain()` after an action shows both the `Final Plan` and
the `Initial Plan`; `join_strategy(df)` runs the query and reports the final one.

In [ ]:
big_join = orders.join(customers, "customer_id")
print("initial plan:       ", re.findall(r"BroadcastHashJoin|SortMergeJoin", "\n".join(plan_lines(big_join))))
print("final plan (AQE):   ", join_strategy(big_join))
with no_auto_broadcast():
    print("broadcast disabled: ", join_strategy(orders.join(customers, "customer_id")))
    print("full outer:         ", join_strategy(orders.join(customers, "customer_id", "full")))

### Broadcast hash join

Spark collects the small side to the driver, sends a copy to every executor, and builds a hash table from it. Each
task of the large side then probes the hash table locally, so the large side is **never shuffled**. It is the
fastest strategy when one side is small.

| control | effect |
| --- | --- |
| `F.broadcast(df)` | hint: broadcast this DataFrame |
| `df.hint("broadcast")` | the same as a DataFrame hint |
| `/*+ BROADCAST(t) */` | the same in SQL |
| `spark.sql.autoBroadcastJoinThreshold` | size below which Spark broadcasts automatically (`-1` disables) |
| `spark.sql.adaptive.autoBroadcastJoinThreshold` | the threshold AQE uses at run time |

**Notes:** the whole small side must fit in the driver and in every executor's memory; broadcasting a table that
is "small" by estimate but large in reality causes driver out-of-memory errors. Size estimates after filters and
UDFs can be wrong; check the plan.

In [ ]:
with no_auto_broadcast():
    hinted = orders.join(F.broadcast(customers), "customer_id")
    print("F.broadcast:", join_strategy(hinted))
    print("df.hint:    ", join_strategy(orders.join(customers.hint("broadcast"), "customer_id")))

### Sort-merge join

Both sides are shuffled so rows with the same key land in the same partition (an `Exchange hashpartitioning` on the
key), each partition is sorted by the key, and the two sorted streams are merged. It handles any size of input
because nothing needs to fit in memory at once, which is why it is the default for large equi-joins.

**Notes:** the cost is two shuffles and two sorts. If both inputs are already partitioned and sorted by the join key
(bucketed tables, or the output of a previous step on the same key), Spark can skip those steps. Skewed keys make
one partition much larger than the rest ([skewed joins](#skewed-joins)).

In [ ]:
with no_auto_broadcast():
    smj = orders.join(customers, "customer_id")
    print([line.strip()[:70] for line in plan_lines(smj) if re.search(r"SortMergeJoin|Exchange|Sort \[", line)])

### Shuffle hash join

Both sides are shuffled by the key, then the **smaller side of each partition** is built into a hash table and the
other side probes it. It skips the sort that a sort-merge join needs, so it can be faster when one side is
noticeably smaller, but each partition's build side must fit in memory.

**Use it for:** joins where one side is too big to broadcast but much smaller than the other. Request it with
`df.hint("shuffle_hash")` or `/*+ SHUFFLE_HASH(t) */`.

In [ ]:
with no_auto_broadcast():
    print(join_strategy(orders.join(customers.hint("shuffle_hash"), "customer_id")))

### Broadcast nested loop join and Cartesian product

Without an equality condition, Spark compares every left row with every right row. If one side is small enough it
is broadcast and each task loops over it (**broadcast nested loop join**); otherwise Spark builds a **Cartesian
product**. Both cost `left_rows × right_rows` comparisons.

**Notes:** this is fine for a few thousand rows against a small band table, and disastrous at scale. Add an equality
on a coarse key (same customer, same date bucket) to turn the join back into a hash or sort-merge join and leave
only the range check per key.

In [ ]:
in_band = (orders.amount >= bands.low) & (orders.amount < bands.high)
print("no size estimate for bands:", join_strategy(orders.join(bands, in_band)))
print("bands broadcast:           ", join_strategy(orders.join(F.broadcast(bands), in_band)))

### Join hints

Hints override Spark's choice when you know better than its size estimates. When both sides carry hints, Spark
prefers them in the order `BROADCAST` > `MERGE` > `SHUFFLE_HASH` > `SHUFFLE_REPLICATE_NL`.

| DataFrame | SQL | strategy |
| --- | --- | --- |
| `F.broadcast(df)` / `df.hint("broadcast")` | `/*+ BROADCAST(t) */` (also `BROADCASTJOIN`, `MAPJOIN`) | broadcast hash join |
| `df.hint("merge")` | `/*+ MERGE(t) */` (also `SHUFFLE_MERGE`, `MERGEJOIN`) | sort-merge join |
| `df.hint("shuffle_hash")` | `/*+ SHUFFLE_HASH(t) */` | shuffle hash join |
| `df.hint("shuffle_replicate_nl")` | `/*+ SHUFFLE_REPLICATE_NL(t) */` | Cartesian product |

**Notes:** a hint that cannot apply (broadcasting the preserved side of an outer join, a hash join without an
equality) is ignored, with a warning in the logs. Re-check hints when data sizes change.

In [ ]:
print("merge hint:", join_strategy(orders.join(customers.hint("merge"), "customer_id")))
q = spark.sql("SELECT /*+ SHUFFLE_HASH(c) */ o.order_id, c.name FROM {o} o JOIN {c} c ON o.customer_id = c.customer_id", o=orders, c=customers)
with no_auto_broadcast():
    print("SQL hint:  ", join_strategy(q))

## Skewed joins

### AQE skew join handling

When a few keys hold most of the rows (one huge customer, a `null` or `"unknown"` key), the partitions for those keys
are much larger than the rest and one task runs far longer than the others. With AQE enabled, a sort-merge join
detects partitions that are both larger than `skewedPartitionFactor` times the median and larger than
`skewedPartitionThresholdInBytes`, splits them into smaller pieces and replicates the matching rows of the other side.

| setting | default | meaning |
| --- | --- | --- |
| `spark.sql.adaptive.enabled` | true | AQE on |
| `spark.sql.adaptive.skewJoin.enabled` | true | split skewed partitions in sort-merge joins |
| `spark.sql.adaptive.skewJoin.skewedPartitionFactor` | 5 | how many times the median a partition must be |
| `spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes` | 256 MB | minimum size to count as skewed |

**Notes:** the split shows up as `isSkew=true` / `SkewJoin` in the final adaptive plan and in the Spark UI. It does
not apply to broadcast joins, which have no shuffle to split.

In [ ]:
for key in ["enabled", "skewJoin.enabled", "skewJoin.skewedPartitionFactor", "skewJoin.skewedPartitionThresholdInBytes"]:
    print(f"spark.sql.adaptive.{key} =", spark.conf.get(f"spark.sql.adaptive.{key}"))

### Salting a hot key

When AQE cannot help (older Spark, or skew in an aggregation rather than a join), spread a hot key manually:

1. On the large side, add a random **salt** from 0 to N-1.
2. On the small side, **replicate** each row N times with every salt value (`explode(sequence(0, N-1))`).
3. Join on `(key, salt)`, so the hot key is split across N partitions.

**Notes:** the small side grows N times, so salt only the hot keys if the small side is large, or choose a small N.
The result must equal the unsalted join; verify it.

In [ ]:
N = 4
clicks = spark.range(2000).select(F.when(F.col("id") < 1800, 1).otherwise(F.col("id") % 8 + 2).alias("customer_id"))  # 90% of rows are customer 1
salted_facts = clicks.withColumn("salt", (F.rand(seed=1) * N).cast("int"))
salted_dim = customers.withColumn("salt", F.explode(F.sequence(F.lit(0), F.lit(N - 1))))
salted = salted_facts.join(salted_dim, ["customer_id", "salt"]).groupBy("name").count()
plain = clicks.join(customers, "customer_id").groupBy("name").count()
print("same result:", salted.exceptAll(plain).count() == 0 and plain.exceptAll(salted).count() == 0)
salted_facts.where("customer_id = 1").groupBy("salt").count().orderBy("salt").show()

## Joins in Spark SQL

### SQL join syntax

Every DataFrame join has a SQL form. Spark SQL adds `LEFT SEMI JOIN` and `LEFT ANTI JOIN` as join types, supports
`USING (cols)` (one copy of each key, like `on=` names) and `NATURAL JOIN` (all same-named columns, best avoided
because a new column silently changes the condition), and takes hints in `/*+ ... */` comments.

```sql
SELECT o.order_id, c.name
FROM orders o
LEFT JOIN customers c USING (customer_id)

SELECT * FROM customers c LEFT ANTI JOIN orders o ON o.customer_id = c.customer_id

SELECT /*+ BROADCAST(p) */ i.order_id, p.category
FROM order_items i LEFT JOIN products p ON i.sku = p.sku
```

**Notes:** `NOT IN (subquery)` returns no rows at all if the subquery produces a null; `NOT EXISTS` and `LEFT ANTI
JOIN` do not have that problem.

In [ ]:
spark.sql("""
    SELECT c.customer_id, c.name, count(o.order_id) AS orders
    FROM {c} c LEFT JOIN {o} o USING (customer_id)
    GROUP BY c.customer_id, c.name
    ORDER BY c.customer_id
""", c=customers, o=orders).show()

with_null = spark.createDataFrame([(1,), (None,)], "customer_id int")
print("NOT IN with a null:", spark.sql("SELECT count(*) FROM {c} WHERE customer_id NOT IN (SELECT customer_id FROM {n})", c=customers, n=with_null).first()[0])
print("LEFT ANTI JOIN:   ", spark.sql("SELECT count(*) FROM {c} c LEFT ANTI JOIN {n} n ON c.customer_id = n.customer_id", c=customers, n=with_null).first()[0])

## Performance checklist

| check | why |
| --- | --- |
| Filter and select columns **before** the join | less data to shuffle; Spark pushes some filters down itself, but not through every operator |
| Broadcast small lookups explicitly when estimates are wrong | avoids shuffling the large side |
| Make sure dimension keys are **unique** | prevents row multiplication |
| Join keys have the **same type** on both sides | avoids implicit casts and lost matches |
| Avoid UDFs and functions on join keys | precompute the key as a column so Spark can hash or sort it |
| Add an equality to range joins | turns a nested loop into a hash or sort-merge join |
| Watch for skewed keys (`null`, `unknown`, one giant customer) | filter or salt them; let AQE split skewed partitions |
| Reuse a partitioning when joining the same large table several times | `repartition(key)` once, or bucket the table |
| Check the plan (`explain`) for `BroadcastHashJoin`, `SortMergeJoin`, `CartesianProduct` | confirms the strategy you expect |
| Count rows before and after the join in tests | catches lost and duplicated rows early |

**Next:** practise these entries in [PySpark practice 9: joins](pyspark-practice-09-joins.ipynb).